### 1.0. Библиотеки

In [1]:
!pip uninstall -y scikit-learn # удалим более старую версию библиотеки
!pip install scikit-learn # установим версию поновее

Found existing installation: scikit-learn 1.8.0
Uninstalling scikit-learn-1.8.0:
  Successfully uninstalled scikit-learn-1.8.0
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)


In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

### 1.1. Датасет

#### Описание датасета

Возьмем датасет [Possum Regression](https://www.kaggle.com/datasets/abrambeyer/openintro-possum).

В этой задаче нужно предсказать возраст оппосума по имеющимся о нем данным:
- **Таргет**: **age** - Возраст
- **case** - Номер наблюдения
- **site** - Номер участка, на котором был отловлен опоссум
- **Pop** - Поселение, либо Vic (Виктория), либо другое (Новый Южный Уэльс или Квинсленд)
- **sex** - Пол
- **hdlngth** - Длина головы, в мм.
- **skullw** - Ширина черепа, в мм.
- **totlngth** 	- Общая длина, в см.
- **taill** 	- Длина хвоста, в см.
- **footlgth** 	- Длина стопы
- **earconch** 	- Длина ушной раковины
- **eye** 	- Расстояние от медиального канта до латерального канта правого глаза
- **chest** 	- Обхват груди (в см)
- **belly** - Обхват живота (в см)

#### Описание задачи

In [5]:
df = pd.read_csv("D:\\6 семестр\\ML\\lab3\\possum.csv")

In [6]:
df.head()

,case,site,Pop,sex,age,hdlngth,skullw,totlngth,taill,footlgth,earconch,eye,chest,belly
0,1,1,Vic,m,8.0,94.1,60.4,89.0,36.0,74.5,54.5,15.2,28.0,36.0
1,2,1,Vic,f,6.0,92.5,57.6,91.5,36.5,72.5,51.2,16.0,28.5,33.0
2,3,1,Vic,f,6.0,94.0,60.0,95.5,39.0,75.4,51.9,15.5,30.0,34.0
3,4,1,Vic,f,6.0,93.2,57.1,92.0,38.0,76.1,52.2,15.2,28.0,34.0
4,5,1,Vic,f,2.0,91.5,56.3,85.5,36.0,71.0,53.2,15.1,28.5,33.0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   case      104 non-null    int64  
 1   site      104 non-null    int64  
 2   Pop       104 non-null    object 
 3   sex       104 non-null    object 
 4   age       102 non-null    float64
 5   hdlngth   104 non-null    float64
 6   skullw    104 non-null    float64
 7   totlngth  104 non-null    float64
 8   taill     104 non-null    float64
 9   footlgth  103 non-null    float64
 10  earconch  104 non-null    float64
 11  eye       104 non-null    float64
 12  chest     104 non-null    float64
 13  belly     104 non-null    float64
dtypes: float64(10), int64(2), object(2)
memory usage: 11.5+ KB


#### Подготовка датасета

Номер наблюдения, номер участка, и место поселения никакого влияния, пол на возраст оппосумов не дает. Поэтому удалим неинформативные для нас признаки.

In [8]:
# удаляем неинформативные признаки
df.drop(columns=['case', 'site', 'Pop', 'sex'], inplace=True)

Посмотрим, в каких колонках есть пропуски:

In [9]:
df.isna().sum()

age         2
hdlngth     0
skullw      0
totlngth    0
taill       0
footlgth    1
earconch    0
eye         0
chest       0
belly       0
dtype: int64

In [10]:
# Удалим строки с пропусками
df.dropna(inplace=True)

X = df.drop(['age'], axis=1)
y = df['age']

In [11]:
df.isna().sum()

age         0
hdlngth     0
skullw      0
totlngth    0
taill       0
footlgth    0
earconch    0
eye         0
chest       0
belly       0
dtype: int64

И разделим нашу выборку на тренировочную и тестовую:

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, shuffle=True, test_size=0.2, random_state=44)

### Реализуем KNN вручную без sklearn

In [13]:
X_train[0]

array([91. , 55. , 84.5, 36. , 72.8, 51.4, 13.6, 27. , 30. ])

In [14]:
X_test[0]

array([94.5, 64.2, 91. , 39. , 66.5, 46.4, 14.4, 30.5, 33. ])

In [18]:
import numpy as np
# Функция расчета расстояния между векторами
def euclidean_distance_numpy(vec1, vec2):
	return np.linalg.norm(vec1 - vec2)

#### **Шаг 2.** Нахождение ближайших соседей

Напишем функцию, которая по тренировочной выборке, тестовому примеру и количеству соседей $k$ находит $k$ ближайших элементов к тестовому элементу.



In [21]:
X_train[:5]

array([[91. , 55. , 84.5, 36. , 72.8, 51.4, 13.6, 27. , 30. ],
       [93.1, 54.8, 90.5, 35.5, 73.2, 53.6, 14.2, 30. , 32. ],
       [88.7, 52. , 83. , 38. , 61.5, 45.9, 14.7, 26. , 34. ],
       [97.6, 61. , 93.5, 40. , 67.9, 44.3, 15.8, 28.5, 32.5],
       [91.6, 56.6, 88.5, 37.5, 64.5, 45.4, 14.9, 27. , 31. ]])

In [22]:
def get_neighbors_numpy(train, test_row, num_neighbors):
  distances = np.linalg.norm(train - test_row, axis=1)
  nearest_neighbor_ids = distances.argsort()[:num_neighbors]
  return nearest_neighbor_ids

In [23]:
get_neighbors_numpy(X_train[:5], X_test[1], 3)

array([4, 2, 3])

#### **Шаг 3.** Получение предсказания для тестовых элементов

Реализуем функцию получения предсказания для тестового набора данных.

In [24]:
def predict_numpy(X_train, X_test, y_train, num_neighbors):

  y_predict = []
  for x_test in X_test:
    nearest_neighbor_ids = get_neighbors_numpy(X_train, x_test, num_neighbors)
    nearest_neighbor_y = y_train[nearest_neighbor_ids]
    pred = np.mean(nearest_neighbor_y)
    y_predict.append(pred)

  return y_predict

In [25]:
y_predict = predict_numpy(X_train[:30], X_test[:5], y_train[:30], num_neighbors = 5)
y_predict

[np.float64(4.4),
 np.float64(3.4),
 np.float64(2.8),
 np.float64(3.4),
 np.float64(3.4)]

### kNN при помощи Sklearn

In [29]:
# Шаг 1. создание модели k = 5
model = KNeighborsRegressor(n_neighbors=5)

# Шаг 2. "обучение" модели (фактически запоминаем данные)
model.fit(X_train, y_train)

# Шаг 3. Предсказание на тестовых данных
y_pred = model.predict(X_test)

In [30]:
y_pred

array([4.4, 4. , 3.2, 5.8, 4. , 4. , 4.6, 2.4, 4.6, 3.8, 2. , 5. , 3. ,
       5.2, 5.8, 5. , 2.2, 2.8, 4.8, 1.6, 3. ])

## 3. Метрики

In [47]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

MSE_train = mean_squared_error(y_train, pred_train) # MSE на обучении
RMSE_train = np.sqrt(MSE_train) # RMSE на обучении
R2_train = r2_score(y_train, pred_train) # R2 на обучении
MAE_train = mean_absolute_error(y_train, pred_train) # MAE на обучении

MSE_test = mean_squared_error(y_test, pred_test) # MSE на тесте
RMSE_test = np.sqrt(MSE_test) # RMSE на тесте
R2_test = r2_score(y_test, pred_test) # R2 на тесте
MAE_test = mean_absolute_error(y_test, pred_test) # MAE на тесте

print(f'MSE на обучении {MSE_train:.2f}') 
print(f'MSE на тесте {MSE_test:.2f}', end='\n\n')

print(f'R2 на обучении {R2_train:.2f}')
print(f'R2 на тесте {R2_test:.2f}', end='\n\n')

print(f'MAE на обучении {MAE_train:.2f}')
print(f'MAE на тесте {MAE_test:.2f}')

MSE на обучении 2.56
MSE на тесте 1.82

R2 на обучении 0.33
R2 на тесте 0.31

MAE на обучении 1.32
MAE на тесте 1.15


### Подбор гиперпараметров для KNN

In [35]:
# Дополнительные импорты для кросс-валидации и поиска
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold, ShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

#### GridSearchCV с KFold

GridSearchCV с KFold (5 фолдов) перебирает все комбинации параметров и находит лучшую.
GridSearchCV выполняет исчерпывающий перебор всех заданных комбинаций гиперпараметров. На каждой комбинации модель обучается на k−1 фолдах и оценивается на оставшемся, процесс повторяется для всех фолдов. Итоговая оценка — среднее значение метрики (в нашем случае отрицательной MSE) по фолдам. Это даёт полную картину, но требует больших вычислительных ресурсов при расширении сетки.

KFold — классический метод, разбивающий данные на k непересекающихся частей; каждая часть по очереди становится валидационной. Оценка стабильна и мало зависит от случайности разбиения.

In [41]:
print("GridSearchCV с KFold (5 блоков):")
param_grid = {
    'n_neighbors': np.arange(1, 21),          # K от 1 до 20
    'weights': ['uniform', 'distance'],       # способ учёта весов соседей
    'metric': ['euclidean', 'manhattan', 'minkowski']  # метрики расстояния
}
knn = KNeighborsRegressor()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(knn, param_grid, cv=kfold,
                           scoring='neg_mean_squared_error',
                           verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Лучшие параметры (GridSearch):", grid_search.best_params_)
print(f'Лучшее значение MSE (среднее по валидации): {-grid_search.best_score_}', end='\n\n')
best_knn_grid = grid_search.best_estimator_
y_pred_grid = best_knn_grid.predict(X_test)
print(f'MSE на тесте (GridSearch): {mean_squared_error(y_test, y_pred_grid)}', end='\n')
print(f'R2 на тесте (GridSearch): {r2_score(y_test, y_pred_grid)}', end='\n')
print(f'MAE на тесте (GridSearch): {mean_absolute_error(y_test,y_pred_grid)}', end='\n')

GridSearchCV с KFold (5 блоков):
Fitting 5 folds for each of 120 candidates, totalling 600 fits
Лучшие параметры (GridSearch): {'metric': 'euclidean', 'n_neighbors': np.int64(9), 'weights': 'uniform'}
Лучшее значение MSE (среднее по валидации): 3.549074074074074

MSE на тесте (GridSearch): 1.7477954144620806
R2 на тесте (GridSearch): 0.3320816483728096
MAE на тесте (GridSearch): 1.1904761904761905


#### RandomizedSearchCV с ShuffleSplit

RandomizedSearchCV с ShuffleSplit (5 случайных разбиений) выбирает случайные комбинации (30 итераций) – это ускоряет поиск при большом пространстве параметров.

RandomizedSearchCV выбирает фиксированное число случайных комбинаций из заданных распределений параметров. Это позволяет быстрее найти близкую к оптимальной область, особенно когда пространство поиска велико, а влияние некоторых параметров нелинейно.

ShuffleSplit генерирует несколько независимых случайных разбиений на train/validation, что полезно, когда важно усреднить результат по разным долям данных, но при этом возможны пересечения объектов в разных разбиениях.

In [45]:
print("RandomizedSearchCV с ShuffleSplit (5 итераций, 20% теста):")
param_dist = {
    'n_neighbors': np.arange(1, 21),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
shuffle_split = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
random_search = RandomizedSearchCV(knn, param_dist, n_iter=30,
                                    cv=shuffle_split,
                                    scoring='neg_mean_squared_error',
                                    verbose=1, n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)

print("Лучшие параметры (RandomizedSearch):", random_search.best_params_)
print(f'Лучшее значение MSE (среднее по валидации): {-random_search.best_score_}', end='\n\n')
best_knn_random = random_search.best_estimator_
y_pred_random = best_knn_random.predict(X_test)
print(f'MSE на тесте (RandomizedSearch): {mean_squared_error(y_test, y_pred_random)}', end='\n')
print(f'R2 на тесте (RandomizedSearch): {r2_score(y_test, y_pred_random)}', end='\n')
print(f'MAE на тесте (RandomizedSearch): {mean_absolute_error(y_test, y_pred_random)}', end='\n')

RandomizedSearchCV с ShuffleSplit (5 итераций, 20% теста):
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Лучшие параметры (RandomizedSearch): {'weights': 'distance', 'n_neighbors': np.int64(13), 'metric': 'manhattan'}
Лучшее значение MSE (среднее по валидации): 3.3016727614476324

MSE на тесте (RandomizedSearch): 1.643188378973117
R2 на тесте (RandomizedSearch): 0.37205712727283835
MAE на тесте (RandomizedSearch): 1.142305306415904


### Сравнение исходной и оптимальных моделей

In [44]:
print("Сравнение метрик на тестовой выборке:")
print("Модель               | MSE на тесте | R2 на тесте")
print("---------------------|--------------|------------")
print(f"Исходная (k=5)       | {mean_squared_error(y_test, y_pred):.4f}       | {r2_score(y_test, y_pred):.4f}")
print(f"GridSearchCV (лучшая)| {mean_squared_error(y_test, y_pred_grid):.4f}       | {r2_score(y_test, y_pred_grid):.4f}")
print(f"RandomizedSearchCV   | {mean_squared_error(y_test, y_pred_random):.4f}       | {r2_score(y_test, y_pred_random):.4f}")

Сравнение метрик на тестовой выборке:
Модель               | MSE на тесте | R2 на тесте
---------------------|--------------|------------
Исходная (k=5)       | 1.8152       | 0.3063
GridSearchCV (лучшая)| 1.7478       | 0.3321
RandomizedSearchCV   | 1.6432       | 0.3721


Из сравнительной таблицы видно, что RandomizedSearchCV в сочетании с ShuffleSplit показал лучшие результаты

## Вывод
В ходе выполнения лабораторной работы была решена задача регрессии — предсказание возраста опоссумов по морфологическим признакам. Для этого использован метод ближайших соседей (KNN), который относится к классу «ленивых» алгоритмов: он не строит модель в явном виде, а запоминает обучающие данные и для каждого нового объекта находит K наиболее похожих примеров, усредняя их целевую переменную. Качество такого подхода критически зависит от выбора гиперпараметров: числа соседей 
K
K, функции расстояния и способа взвешивания соседей.

#### Подготовка данных
Исходный датасет содержал пропуски в признаках age и footlgth, которые были удалены (2 строки). Неинформативные категориальные признаки (case, site, Pop, sex) исключены, так как они не влияют на целевую переменную. После очистки выборка была разделена в соотношении 80/20 с фиксацией random_state для воспроизводимости.

#### Базовая модель
В качестве отправной точки взята модель KNN с K=5, евклидовой метрикой и равными весами соседей. На тестовой выборке она показала MSE = 1.815, R² = 0.306, MAE = 1.15. Низкое значение R² говорит о том, что модель объясняет лишь около 30% дисперсии целевой переменной — это ожидаемо для простого KNN без настройки гиперпараметров.

Подбор гиперпараметров
Для улучшения качества применены два подхода к поиску оптимальных параметров:

#### GridSearchCV полный перебор по сетке значений: 
K от 1 до 20, метрики euclidean, manhattan, minkowski, типы весов uniform и distance. В качестве стратегии кросс-валидации использован KFold с 5 блоками, что гарантирует оценку на всех непересекающихся подмножествах обучающих данных. Лучшая комбинация: 
K = 9
K=9, uniform, euclidean; средняя MSE на валидации составила 3.55, а на тесте — 1.748 (R² = 0.332).

#### RandomizedSearchCV — случайный поиск по 30 итерациям из того же пространства параметров.
Кросс-валидация выполнялась с помощью ShuffleSplit (5 случайных разбиений, 20% данных — валидация). Этот метод быстрее при большом количестве комбинаций. Оптимальные параметры: K=13, distance, manhattan; MSE на валидации = 3.30, на тесте = 1.643 (R² = 0.372).

### Анализ результатов
Обе процедуры подбора улучшили качество по сравнению с базовой моделью. Наилучший результат достигнут с помощью RandomizedSearchCV: снижение MSE на 9.5% и рост R² на 21.6% относительно исходной модели. Интересно, что лучшая модель использует взвешивание по расстоянию (weights='distance') и метрику Манхэттена, что может лучше отражать структуру данных.